In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl

In [3]:
import os
import shutil
import torch
from google.colab import drive
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer

drive.mount('/content/drive')
PROJECT_PATH = "/content/drive/MyDrive/VictorianGPT"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
data_path = f"{PROJECT_PATH}/dialogues/victorian_dataset.json"
dataset = load_dataset('json', data_files=data_path, split='train')
print(f"Loaded {len(dataset)} authentic dialogue pairs.")

Loaded 7096 authentic dialogue pairs.


In [5]:
def format_chatml(example):
    # Basic cleanup
    user_text = example["input"].strip()
    assistant_text = example["output"].strip()

    # Structure for the model
    text = f"<|im_start|>user\n{user_text}\n<|im_end|>\n<|im_start|>assistant\n{assistant_text}\n<|im_end|>"
    return {"text": text}

dataset = dataset.map(format_chatml)
print(dataset[0]['text']) # Preview the first formatted example

<|im_start|>user
She regretted to be under the necessity of keeping me at a distance; but that until she heard from Bessie, and could discover by her own observation, that I was endeavouring in good earnest to acquire a more sociable and childlike disposition, a more attractive and sprightly manner—something lighter, franker, more natural, as it were—she really must exclude me from privileges intended only for contented, happy, little children.
<|im_end|>
<|im_start|>assistant
What does Bessie say I have done?
<|im_end|>


In [12]:
# Cell 5: Load Base Model & Tokenizer (Fixed for T4 GPU)
model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16  # <--- THIS IS THE FIX
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure pad token is set for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [13]:
# Prepare for QLoRA
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Crucial: Delete the old, overfitted adapter from your Drive before retraining
adapter_path = f"{PROJECT_PATH}/victorian_adapter"
shutil.rmtree(adapter_path, ignore_errors=True)
print("Previous adapter weights cleared.")

Previous adapter weights cleared.


In [14]:
training_args = TrainingArguments(
    output_dir="./victorian_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,          # Lowered for real data
    num_train_epochs=1,          # Lowered to prevent memorizing the novels
    weight_decay=0.01,           # Added for regularization
    logging_steps=10,
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit"
)

In [16]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_args,
)

# --- THE PURGE: Forcibly override Qwen's native BFloat16 ---
# Force the core config to acknowledge FP16
model.config.torch_dtype = torch.float16

# Route parameters to their proper data types
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        # Trainable adapter weights MUST be float32 for the GradScaler
        param.data = param.data.to(torch.float32)
    elif param.dtype == torch.bfloat16:
        # Lingering non-trainable base weights become float16
        param.data = param.data.to(torch.float16)

# Scour hidden buffers (like RoPE embeddings) and cast them
for name, buffer in trainer.model.named_buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float16)

print("Data types safely routed. Commencing fine-tuning...")

# Train!
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Data types safely routed. Commencing fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,4.636301
20,4.900862
30,4.116072
40,3.907084
50,3.555178
60,3.406533
70,3.369438
80,3.208973
90,3.085368
100,3.018549


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=887, training_loss=2.8332973639349524, metrics={'train_runtime': 2452.1801, 'train_samples_per_second': 2.894, 'train_steps_per_second': 0.362, 'total_flos': 1.323168299851776e+16, 'train_loss': 2.8332973639349524, 'entropy': 2.6402859006609236, 'num_tokens': 542324.0, 'mean_token_accuracy': 0.45804726225989206, 'epoch': 1.0})

In [17]:
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"VictorianGPT adapter successfully saved to {adapter_path}")

VictorianGPT adapter successfully saved to /content/drive/MyDrive/VictorianGPT/victorian_adapter


In [21]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

user_input = "I am feeling quite fearful of the dark tonight."

# Add a strict system prompt to force the dark, philosophical aesthetic
formatted_prompt = f"""<|im_start|>system
You are a highly educated scholar from the 19th century. Speak with sophisticated English, employing elegant metaphors and dark, philosophical observations about the nature of the world.
<|im_end|>
<|im_start|>user
{user_input}
<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.8, # Slightly higher temperature for more creative metaphors
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

print("Test Output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1])

Test Output:
Do not fear, for you will soon be dead and your fear gone.

